# 03 — Evaluación con RAGAS

Ahora medimos si las técnicas del notebook 02 realmente mejoran el sistema.
Sin métricas, cualquier cambio es una apuesta a ciegas.

**RAGAS** evalúa 2 dimensiones (optimizado para bajo consumo de tokens):

| Métrica | Pregunta | Qué detecta |
|---------|----------|-------------|
| **Faithfulness** | ¿La respuesta viene del contexto? | Alucinaciones |
| **Answer Relevancy** | ¿Responde la pregunta? | Respuestas vagas |

### Optimizaciones de tokens aplicadas
- **Modelo juez**: `llama-3.1-8b-instant` en lugar de 70B (~9× más barato)
- **3 preguntas** en lugar de 5
- **k=3 chunks** por pregunta (antes k=4)
- **Contextos truncados** a 400 chars antes de enviar al juez
- **max_tokens=256** en el LLM juez
- **2 métricas** en lugar de 4

Resultado: ~6× menos tokens vs. configuración original.

**Prerequisito**: haber ejecutado el notebook 01

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

from src.ingestion.embedder import get_embeddings
from src.retrieval.vectorstore import load_vectorstore
from src.generation.chain import build_rag_chain, format_docs
from src.evaluation.ragas_eval import EVAL_QUESTIONS, build_eval_dataset, run_evaluation, print_summary, _truncate_contexts

embeddings = get_embeddings()
vectorstore = load_vectorstore(embeddings)
print('Listo')

## Las preguntas de evaluación

Usamos **3 preguntas** con respuestas de referencia (ground truth) extraídas de los papers.
(Reducidas de 5 a 3 para ahorrar ~40% de tokens en el paso de generación.)

In [ ]:
for i, q in enumerate(EVAL_QUESTIONS):
    print(f'[{i+1}] Q: {q["question"]}')
    print(f'     GT: {q["ground_truth"][:100]}...')
    print()

## Evaluar las 4 estrategias

> **Estimado de tokens**: 3 preguntas × 4 estrategias = 12 llamadas para generar respuestas
> + ~24 llamadas RAGAS con `llama-3.1-8b-instant` (256 tokens max cada una).
> Tarda ~3-5 minutos. El free tier de Groq lo aguanta sin problema.

**Por qué aparecían NaN antes**: El 70B con contextos largos agotaba el rate limit de
Groq y RAGAS no podía parsear las respuestas truncadas. Con el 8B y contextos cortos
esto se resuelve.

In [ ]:
from src.retrieval.strategies import retrieve_hyde, retrieve_rag_fusion, retrieve_with_reranking
import os
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from datasets import Dataset

RAG_PROMPT = ChatPromptTemplate.from_template("""
Responde basándote ÚNICAMENTE en el contexto. Si no está en el contexto, dilo.
Contexto: {context}
Pregunta: {question}
Respuesta (máx. 3 frases):""")

# 70B para generar respuestas (calidad), 8B como juez en run_evaluation (ahorra tokens)
llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0, max_tokens=300,
               api_key=os.getenv('GROQ_API_KEY'))

def evaluate_strategy(name, retrieve_fn):
    """Evalúa una estrategia de retrieval con RAGAS."""
    print(f'\n--- Evaluando: {name} ---')

    rows = []
    for item in EVAL_QUESTIONS:
        q = item['question']
        docs = retrieve_fn(q)
        context = format_docs(docs)
        answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({'context': context, 'question': q})
        rows.append({
            'question': q,
            'answer': answer,
            # Truncar contextos ANTES de enviar a RAGAS para reducir tokens del juez
            'contexts': _truncate_contexts([doc.page_content for doc in docs]),
            'ground_truth': item['ground_truth'],
        })
        print(f'  ✓ {q[:50]}...')

    dataset = Dataset.from_list(rows)
    df = run_evaluation(dataset)  # usa llama-3.1-8b-instant internamente
    summary = print_summary(df, name)
    return summary

# k=3 en todas las estrategias
strategies = {
    'Naive RAG':    lambda q: vectorstore.similarity_search(q, k=3),
    'HyDE':         lambda q: retrieve_hyde(vectorstore, q, k=3),
    'RAG-Fusion':   lambda q: retrieve_rag_fusion(vectorstore, q, k=3),
    'CrossEncoder': lambda q: retrieve_with_reranking(vectorstore, q, k_initial=8, k_final=3),
}

all_results = {}
for name, fn in strategies.items():
    all_results[name] = evaluate_strategy(name, fn)

## Tabla de resultados comparativa

In [ ]:
results_df = pd.DataFrame(all_results).T
results_df.index.name = 'Strategy'

print('\n=== RESULTADOS RAGAS ===')
print(results_df.to_string())

print('\n=== MEJOR POR MÉTRICA ===')
for col in results_df.columns:
    valid = results_df[col].dropna()
    if not valid.empty:
        best = valid.idxmax()
        val = valid.max()
        print(f'  {col:<25} → {best} ({val:.3f})')
    else:
        print(f'  {col:<25} → sin datos válidos')

In [ ]:
# Guarda resultados
results_df.to_csv(ROOT / 'data' / 'evaluation_results.csv')
print('Resultados guardados en data/evaluation_results.csv')
results_df

## Interpretación

Con estos resultados puedes responder preguntas concretas:

- *¿Cuánto mejoró el Faithfulness con HyDE respecto al Naive RAG?*
- *¿Qué técnica reduce más las alucinaciones?*
- *¿Vale la pena el coste extra de RAG-Fusion?*

→ En el **notebook 04** construimos la app Streamlit con estas métricas visibles.